# IBM Employee Attrition Analysis

**Dataset:** [IBM HR Analytics Employee Attrition & Performance](https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset) (Kaggle)  
**Tools:** Python · pandas · seaborn · matplotlib · scikit-learn

---

## Project Overview

This project uses the IBM HR Analytics benchmark dataset (1,470 employees, 35 variables) to examine attrition patterns and compare several analytical approaches. Completed as graduate coursework in people analytics and machine learning, the project is intended to demonstrate workflow design, interpretation, and communication using public data.

**Question:** Which employee characteristics and work conditions are most associated with attrition, and how should those patterns be explored before modeling?

## This Notebook: Data Preparation and EDA

This notebook covers the first phase of the analysis:

- **Data overview** — 35 variables across 1,470 employees, no missing values
- **Data cleaning** — removing zero-variance columns and preparing variables for analysis
- **Class imbalance check** — attrition is the minority outcome
- **Exploratory analysis** — visualizing patterns across demographic, job, and compensation variables


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import statsmodels.api as sm 
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier 
from sklearn.ensemble import RandomForestClassifier 
from sklearn.model_selection import cross_val_score
from sklearn.feature_selection import mutual_info_classif
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import * # import all libraries under sklearn.metrics
from sklearn import preprocessing
from sklearn.decomposition import PCA
from sklearn.metrics import *
plt.rc("font", size=14)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
sns.set(style="white")
sns.set(style="whitegrid", color_codes=True)





# ----
# See #1137: this allows compatibility for scikit-learn >= 0.24
import sklearn.utils
from sklearn.utils import safe_indexing
# except ImportError:
# from sklearn.utils import _safe_indexing

# Oversample and plot imbalanced dataset with ADASYN
from collections import Counter
from imblearn.over_sampling import ADASYN
from imblearn.under_sampling import RandomUnderSampler
from matplotlib import pyplot
from numpy import where


%matplotlib inline


# 2 functions to print out metrics (written by us, not in sklearn)

# define a function for calculating the metric to be used later 
# takes in 2 inputs: Y_pred, Y_true
# and uses them to calculate metrics using functions in sklearn
def classification_metrics(Y_pred, Y_true):
    acc = accuracy_score(Y_true, Y_pred)
    precision = precision_score(Y_true, Y_pred)
    recall = recall_score(Y_true, Y_pred)
    f1score = f1_score(Y_true, Y_pred)
    auc = roc_auc_score(Y_true, y_pred)

    # the function's outputs are the 5 variables below
    return acc, precision, recall, f1score, auc

# define a function for printing the metrics using inputs: classifierName, Y_pred, Y_true
# e.g. inputs can be: 'Logistic Regression', y_pred, y_test
# inside the function, we do something with the inputs (e.g. run classification_metrics on the inputs)
# classification_metrics is antoher function we wrote above
def display_metrics(classifierName, Y_pred, Y_true):
    print ("______________________________________________")
    print ("Model: "+classifierName)
    acc, precision, recall, f1score, auc = classification_metrics(Y_pred, Y_true)
    # returns 5 vars: acc, precision, recall, f1score, auc
    # print them below
    print ("Accuracy: "+str(acc))
    print ("Precision: "+str(precision))
    print ("Recall: "+str(recall))
    print ("F1-score: "+str(f1score))
    print ("AUC: "+str(auc))
    print ("______________________________________________")
    print ("")






In [ ]:
#import data 
df = pd.read_csv('HR_Attrition_IBM.csv')
df

df0 = df

## Data Definition

### Variables in Data

####### Age                          
####### Attrition                   
####### BusinessTravel              
####### DailyRate                   
####### Department                   
####### DistanceFromHome             
####### Education                    
####### EducationField               
####### EmployeeCount                
####### EmployeeNumber               
####### EnvironmentSatisfaction      
####### Gender                       
####### HourlyRate                   
######## JobInvolvement               
######## JobLevel                     
######## JobRole                      
######## JobSatisfaction              
######## MaritalStatus                
######## MonthlyIncome                
######## MonthlyRate                  
######## NumCompaniesWorked           
######## Over18                       
######## OverTime                     
######## PercentSalaryHike            
######## PerformanceRating            
######## RelationshipSatisfaction    
######## StandardHours                
######## StockOptionLevel             
######## TotalWorkingYears            
######## TrainingTimesLastYear        
######## WorkLifeBalance              
######## YearsAtCompany               
######## YearsInCurrentRole           
######## YearsSinceLastPromotion      
######## YearsWithCurrManager         

## Columns and Observations

# Pre - Processing

In [ ]:
df.columns
df.shape # Columns and rows 
#1470 Rows X 35 Columns (variables)
df.info()

## Missing Data


In [ ]:
print(df.isnull()) # for each cell, print True/False (True = missing Data)

#df[df['player'].isnull()] # filtering that keeps rows with missing df.purpose for a variable

#df.dropna() # drop any row with ANY missing value for any feature in a row. 
df #NO MISSING DATA
print(len(df)) # NO MISSING DATA
print(df.shape)


# NO MISSING DATA



In [ ]:
#see number of rows of a dataframe or variable 
#print(len(df)) # number of rows 
#print(len(df.VARIABLE)) # number of rows in a var (purpose)


# Needless Data

In [ ]:
# ---Removing Variable - "Over18" - all enteries are over 18--#
# df.Over18
df.Over18.unique()
df = df.drop('Over18', axis=1) # axis=1 indicates that 'new' is a column 



# ---Removing Variable - "StandardHours" - all enteries are 80---#
# df.StandardHours
df.StandardHours.unique()
df = df.drop('StandardHours', axis=1) # axis=1 indicates that 'new' is a column 



# [1470 rows x 33 columns]

In [ ]:
# Converting Independent Variable DataTypes 


##Boolean/Categorical Data(attrition, Gender, Overtime)

##---OverTime----##
# df['OverTime'] = df.OverTime.astype('bool') #convert var 'date'to a date data type in our dataframe 'df'
df.loc[df["OverTime"] == "Yes", "OverTime"] = 1
df.loc[df["OverTime"] == "No", "OverTime"] = 0
##---Gender----##
# df['Gender'] = df.Gender.astype('bool') #convert var 'date'to a date data type in our dataframe 'df'
df.loc[df["Gender"] == "Male", "Gender"] = 1
df.loc[df["Gender"] == "Female", "Gender"] = 0


##Nominal Data/Ordinal Data/ Interval Data/ Ratio Data (PerformanceRating, etc.))

# print(df.info())
# print(df)



#[1470 rows x 33 columns]



In [ ]:
df['Gender'], df0['Gender']

In [ ]:
df['OverTime'], df0['OverTime']

### Assumption #1: The Response/Dependent Variable is Binary -- ## Data Type

In [ ]:
# Converting Dependent Variable DataType

##Boolean/Categorical Data(attrition, Gender, Overtime)

##---Attrition----##
# df['Attrition'] = df.Attrition.astype('bool') #convert var 'date'to a date data type in our dataframe 'df'

# df.loc[df["Attrition"] == "Yes", "Attrition"] = 1
# df.loc[df["Attrition"] == "No", "Attrition"] = 0
df["Attrition"] = np.where(df["Attrition"] == "No", 0, 1)

##Nominal Data/Ordinal Data/ Interval Data/ Ratio Data (PerformanceRating, etc.))

# print(df.info())
# print(df)

df
#[1470 rows x 33 columns]


In [ ]:
df['Attrition'] = df['Attrition'].astype(bool)          # Transform integer to boolean


In [ ]:
df['Attrition'], df0['Attrition']


In [ ]:
df.Attrition.value_counts()



In [ ]:
count_no_At = len(df[df['Attrition']==0])
count_At = len(df[df['Attrition']==1])
pct_of_no_At = count_no_At/(count_no_At+count_At)
print("percentage of no Attrition is", pct_of_no_At*100)
pct_of_At = count_At/(count_no_At+count_At)
print("percentage of Attrition", pct_of_At*100)

In [ ]:
#check False/ No Attrition
1233/(1233+237)

# df.Attrition



In [ ]:
sns.countplot(x='Attrition', data=df, palette='hls')
plt.show()
# plt. savefig('count plot')

In [ ]:
total = float(len(df))
ax = sns.countplot(x='Attrition', data=df, palette='hls')
plt.title('Attrition Class Distribution', fontsize=20)
for p in ax.patches:
    ax.annotate('{:.1f}'.format(p.get_height()), (p.get_x()+0.25, p.get_height()+0.01), fontsize=13)
for p in ax.patches:
    percentage = '{:.1f}%'.format(100 * p.get_height()/total)
    x = p.get_x() + p.get_width()
    y = p.get_height()
    ax.annotate(percentage, (x, y),ha='center', fontsize=13)
plt.show()



In [ ]:
# sns.set(style="whitegrid")
# plt.figure(figsize=(8,5))
# total = float(len(train_df))
# ax = sns.countplot(x="event", hue="event", data=train_df)
# plt.title('Data provided for each event', fontsize=20)
# for p in ax.patches:
#     percentage = '{:.1f}%'.format(100 * p.get_height()/total)
#     x = p.get_x() + p.get_width()
#     y = p.get_height()
#     ax.annotate(percentage, (x, y),ha='center')
# plt.show()


In [ ]:
##Our classes are imbalanced, and the ratio of no-subscription to subscription instances is 89:11. 
##Before we go ahead to balance the classes, let’s do some more exploration.



In [ ]:
df.groupby('Attrition').mean()



In [ ]:
df.groupby('Age').sum().sort_values(by='Attrition', ascending = False) 



In [ ]:
table=pd.crosstab(df.Age,df.Attrition)
table.div(table.sum(1).astype(float), axis=0).plot(kind='bar', stacked=True)
plt.title('Attrition by Age')
plt.xlabel('Age')
plt.ylabel('Proportion of Attrition')
# plt.savefig('YearsSinceLastPromotion_vs_Attrition_stack')



## Data Prep: Decision Tree

### 1. Unbalanced DV

In [ ]:
# Oversample and plot imbalanced dataset with ADASYN
from collections import Counter
from sklearn.datasets import make_classification
from imblearn.over_sampling import ADASYN
from matplotlib import pyplot
from numpy import where

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)
#Apply Over Sampling
print('Before Oversampling')
print(sorted(Counter(y_train).items()))
X_train, y_train = SMOTE().fit_resample(X_train, y_train)
print('After Oversampling')
print(sorted(Counter(y_train).items()))
#Standard Scaler
#scaler = StandardScaler()  
#scaler.fit(X_train)  
#X_train = scaler.transform(X_train)  
#X_test = scaler.transform(X_test)  

### Assumption #2: The Observations are Independent

In [ ]:
# # Histogram and density curve of total bills overlapping
# sns.distplot(df)



### Assumption #3: There is No Multicollinearity Among Explanatory Variables

In [ ]:
# Matrix form for correlation data
df.corr()

In [ ]:
# Correlations among 3 variables color coded on a heat map
# see colors insetad of values
sns.heatmap(df.corr())

In [ ]:
sns.pairplot(df) # automatically use continuous vars


### Assumption #4: There are No Extreme Outliers

### Assumption #5: There is a Linear Relationship Between Explanatory Variables and the Logit of the Response Variable

### Assumption #6: The Sample Size is Sufficiently Large

# Data Exploration

## Data Distribution

In [ ]:
%matplotlib inline
pd.crosstab(df.DailyRate,df.Attrition).plot(kind='bar')
plt.title('Attrition Frequency by DailyRate ')
plt.xlabel('DailyRate')
plt.ylabel('Frequency of Attrition')
plt.savefig('Attrition_fre_DailyRate')

In [ ]:
print(df.columns)
#df.groupby('Gender').mean()

In [ ]:
%matplotlib inline
pd.crosstab(df.Education,df.Attrition).plot(kind='bar')
plt.title('Attrition Frequency by Job Role ')
plt.xlabel('JobRole')
plt.ylabel('Frequency of Attrition')
plt.savefig('Attrition_fre_JobRole')

In [ ]:
%matplotlib inline
pd.crosstab(df.JobRole,df.Attrition).plot(kind='bar')
plt.title('Attrition Frequency by Job Role ')
plt.xlabel('JobRole')
plt.ylabel('Frequency of Attrition')
plt.savefig('Attrition_fre_JobRole')

In [ ]:
%matplotlib inline
pd.crosstab(df.JobRole,df.Attrition).plot(kind='bar')
plt.title('Attrition Frequency by Job Role ')
plt.xlabel('JobRole')
plt.ylabel('Frequency of Attrition')
plt.savefig('Attrition_fre_JobRole')

In [ ]:
table=pd.crosstab(df.BusinessTravel,df.Attrition)
table.div(table.sum(1).astype(float), axis=0).plot(kind='bar', stacked=True)
plt.title('Stacked Bar Chart of Business Travel vs Attrition')
plt.xlabel('Business Travel')
plt.ylabel('Proportion of Customers')
plt.savefig('BusinessTravel_vs_Attrition_stack')

In [ ]:
table=pd.crosstab(df.Department,df.Attrition)
table.div(table.sum(1).astype(float), axis=0).plot(kind='bar', stacked=True)
plt.title('Stacked Bar Chart of Department vs Attrition')
plt.xlabel('Department')
plt.ylabel('Proportion of Customers')
plt.savefig('Department_vs_Attrition_stack')

In [ ]:
table=pd.crosstab(df.Education,df.Attrition)
table.div(table.sum(1).astype(float), axis=0).plot(kind='bar', stacked=True)
plt.title('Stacked Bar Chart of Education vs Attrition')
plt.xlabel('Education')
plt.ylabel('Proportion of Customers')
plt.savefig('Education_vs_Attrition_stack')

In [ ]:
table=pd.crosstab(df.EducationField,df.Attrition)
table.div(table.sum(1).astype(float), axis=0).plot(kind='bar', stacked=True)
plt.title('Stacked Bar Chart of EducationField vs Attrition')
plt.xlabel('EducationField')
plt.ylabel('Proportion of Customers')
plt.savefig('EducationField_vs_Attrition_stack')

In [ ]:
table=pd.crosstab(df.JobInvolvement,df.Attrition)
table.div(table.sum(1).astype(float), axis=0).plot(kind='bar', stacked=True)
plt.title('Stacked Bar Chart of JobInvolvement vs Attrition')
plt.xlabel('JobInvolvement')
plt.ylabel('Proportion of Customers')
plt.savefig('JobInvolvement_vs_Attrition_stack')

In [ ]:
table=pd.crosstab(df.EnvironmentSatisfaction,df.Attrition)
table.div(table.sum(1).astype(float), axis=0).plot(kind='bar', stacked=True)
plt.title('Stacked Bar Chart of EnvironmentSatisfaction vs Attrition')
plt.xlabel('EnvironmentSatisfaction')
plt.ylabel('Proportion of Customers')
plt.savefig('EnvironmentSatisfaction_vs_Attrition_stack')

In [ ]:
table=pd.crosstab(df.Gender,df.Attrition)
table.div(table.sum(1).astype(float), axis=0).plot(kind='bar', stacked=True)
plt.title('Stacked Bar Chart of Gender vs Attrition')
plt.xlabel('Gender')
plt.ylabel('Proportion of Customers')
plt.savefig('Gender_vs_Attrition_stack')

In [ ]:
table=pd.crosstab(df.MaritalStatus,df.Attrition)
table.div(table.sum(1).astype(float), axis=0).plot(kind='bar', stacked=True)
plt.title('Stacked Bar Chart of MaritalStatus vs Attrition')
plt.xlabel('MaritalStatus')
plt.ylabel('Proportion of Customers')
plt.savefig('MaritalStatus_vs_Attrition_stack')

In [ ]:
table=pd.crosstab(df.NumCompaniesWorked,df.Attrition)
table.div(table.sum(1).astype(float), axis=0).plot(kind='bar', stacked=True)
plt.title('Stacked Bar Chart of NumCompaniesWorked vs Attrition')
plt.xlabel('NumCompaniesWorked')
plt.ylabel('Proportion of Customers')
plt.savefig('NumCompaniesWorked_vs_Attrition_stack')

In [ ]:
table=pd.crosstab(df.OverTime,df.Attrition)
table.div(table.sum(1).astype(float), axis=0).plot(kind='bar', stacked=True)
plt.title('Stacked Bar Chart of OverTime vs Attrition')
plt.xlabel('OverTime')
plt.ylabel('Proportion of Customers')
plt.savefig('OverTime_vs_Attrition_stack')

In [ ]:
table=pd.crosstab(df.PercentSalaryHike,df.Attrition)
table.div(table.sum(1).astype(float), axis=0).plot(kind='bar', stacked=True)
plt.title('Stacked Bar Chart of PercentSalaryHike vs Attrition')
plt.xlabel('PercentSalaryHike')
plt.ylabel('Proportion of Customers')
plt.savefig('PercentSalaryHike_vs_Attrition_stack')

In [ ]:
table=pd.crosstab(df.PerformanceRating,df.Attrition)
table.div(table.sum(1).astype(float), axis=0).plot(kind='bar', stacked=True)
plt.title('Stacked Bar Chart of PerformanceRating vs Attrition')
plt.xlabel('PerformanceRating')
plt.ylabel('Proportion of Customers')
plt.savefig('PerformanceRating_vs_Attrition_stack')

In [ ]:
table=pd.crosstab(df.RelationshipSatisfaction,df.Attrition)
table.div(table.sum(1).astype(float), axis=0).plot(kind='bar', stacked=True)
plt.title('Stacked Bar Chart of RelationshipSatisfaction vs Attrition')
plt.xlabel('RelationshipSatisfaction')
plt.ylabel('Proportion of Customers')
plt.savefig('RelationshipSatisfaction_vs_Attrition_stack')

In [ ]:
table=pd.crosstab(df.StandardHours,df.Attrition)
table.div(table.sum(1).astype(float), axis=0).plot(kind='bar', stacked=True)
plt.title('Stacked Bar Chart of StandardHours vs Attrition')
plt.xlabel('StandardHours')
plt.ylabel('Proportion of Customers')
plt.savefig('StandardHours_vs_Attrition_stack')

In [ ]:
table=pd.crosstab(df.TotalWorkingYears,df.Attrition)
table.div(table.sum(1).astype(float), axis=0).plot(kind='bar', stacked=True)
plt.title('Stacked Bar Chart of TotalWorkingYears vs Attrition')
plt.xlabel('TotalWorkingYears')
plt.ylabel('Proportion of Customers')
plt.savefig('TotalWorkingYears_vs_Attrition_stack')

In [ ]:
table=pd.crosstab(df.YearsAtCompany,df.Attrition)
table.div(table.sum(1).astype(float), axis=0).plot(kind='bar', stacked=True)
plt.title('Stacked Bar Chart of YearsAtCompany vs Attrition')
plt.xlabel('YearsAtCompany')
plt.ylabel('Proportion of Employees')
plt.savefig('YearsAtCompany_vs_Attrition_stack')

In [ ]:
table=pd.crosstab(df.StockOptionLevel,df.Attrition)
table.div(table.sum(1).astype(float), axis=0).plot(kind='bar', stacked=True)
plt.title('Stacked Bar Chart of StockOptionLevel vs Attrition')
plt.xlabel('StockOptionLevel')
plt.ylabel('Proportion of Customers')
plt.savefig('StockOptionLevel_vs_Attrition_stack')

In [ ]:
table=pd.crosstab(df.TrainingTimesLastYear,df.Attrition)
table.div(table.sum(1).astype(float), axis=0).plot(kind='bar', stacked=True)
plt.title('Stacked Bar Chart of TrainingTimesLastYear vs Attrition')
plt.xlabel('TrainingTimesLastYear')
plt.ylabel('Proportion of Customers')
plt.savefig('TrainingTimesLastYear_vs_Attrition_stack')

In [ ]:
table=pd.crosstab(df.WorkLifeBalance,df.Attrition)
table.div(table.sum(1).astype(float), axis=0).plot(kind='bar', stacked=True)
plt.title('Stacked Bar Chart of WorkLifeBalance vs Attrition')
plt.xlabel('WorkLifeBalance')
plt.ylabel('Proportion of Customers')
plt.savefig('WorkLifeBalance_vs_Attrition_stack')

In [ ]:
table=pd.crosstab(df.YearsInCurrentRole,df.Attrition)
table.div(table.sum(1).astype(float), axis=0).plot(kind='bar', stacked=True)
plt.title('Stacked Bar Chart of YearsInCurrentRole vs Attrition')
plt.xlabel('YearsInCurrentRole')
plt.ylabel('Proportion of Customers')
plt.savefig('YearsInCurrentRole_vs_Attrition_stack')

In [ ]:
table=pd.crosstab(df.YearsSinceLastPromotion,df.Attrition)
table.div(table.sum(1).astype(float), axis=0).plot(kind='bar', stacked=True)
plt.title('Stacked Bar Chart of YearsSinceLastPromotion vs Attrition')
plt.xlabel('YearsSinceLastPromotion')
plt.ylabel('Proportion of Customers')
plt.savefig('YearsSinceLastPromotion_vs_Attrition_stack')

In [ ]:
table=pd.crosstab(df.YearsWithCurrManager,df.Attrition)
table.div(table.sum(1).astype(float), axis=0).plot(kind='bar', stacked=True)
plt.title('Stacked Bar Chart of YearsWithCurrManager vs Attrition')
plt.xlabel('YearsWithCurrManager')
plt.ylabel('Proportion of Customers')
plt.savefig('YearsWithCurrManager_vs_Attrition_stack')

In [ ]:
df.columns

In [ ]:
df.Age.hist()
plt.title('Histogram of Age')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.savefig('hist_age')

In [ ]:
df.DailyRate.hist()
plt.title('Histogram of DailyRate')
plt.xlabel('DailyRate')
plt.ylabel('Frequency')
plt.savefig('hist_DailyRate')

In [ ]:
df.columns

In [ ]:
# # Feel free to explore other parameters 
# # Distribution of total bills by day and sex of server
# # hue is a third dimension



# plt.figure(figsize=(15,6))
# sns.boxplot(x="day", y="total_bill", hue="sex", data=tips, palette="rainbow")



In [ ]:
df.info()
df



# Logistic Regressoin 

In [ ]:
# Replicate original data so not to overwrite it 
df1 = df.copy()
dfs = df.copy()



In [ ]:
# create dummy variables for each country using function pd.get_dummies() for variable 'Country'
# add a prefix to names of dummies using 'prefix='Country'
# embed pd.get_dummies(ad_data2['Country'], prefix='Country') inside pd.conat()

# another way:
# countries = pd.get_dummies(ad_data2['Country'], prefix='Country')
# ad_data2 = pd.concat([ad_data2, countries],axis=1)

##-----BusinessTravel------###
df1 = pd.concat([df1, pd.get_dummies(df1['BusinessTravel'], prefix='BusinessTravel')],axis=1)
##-----Department------###
df1 = pd.concat([df1, pd.get_dummies(df1['Department'], prefix='Department')],axis=1)
##-----EducationField------###
df1 = pd.concat([df1, pd.get_dummies(df1['EducationField'], prefix='EducationField')],axis=1)
##-----JobRole------###
df1 = pd.concat([df1, pd.get_dummies(df1['JobRole'], prefix='JobRole')],axis=1)
##-----MaritalStatus------###
df1 = pd.concat([df1, pd.get_dummies(df1['MaritalStatus'], prefix='MaritalStatus')],axis=1)



In [ ]:
### now drop the original 'country' column (you don't need it anymore)


##-----BusinessTravel------###
df1.drop(['BusinessTravel'],axis=1, inplace=True)
##-----Department------###
df1.drop(['Department'],axis=1, inplace=True)

##-----EducationField------###
df1.drop(['EducationField'],axis=1, inplace=True)

##-----JobRole------###
df1.drop(['JobRole'],axis=1, inplace=True)

##-----MaritalStatus------###
df1.drop(['MaritalStatus'],axis=1, inplace=True)


In [ ]:
df1.info()
df1

In [ ]:
df1.columns



In [ ]:
# Create a list of predictor (x) variables: just Age
predictors1 = ['Age', 'DailyRate', 'DistanceFromHome', 'Education',
       'EmployeeCount', 'EmployeeNumber', 'EnvironmentSatisfaction', 'Gender',
       'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction',
       'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'OverTime',
       'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction',
               'StockOptionLevel', 'TotalWorkingYears',
       'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany',
       'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager',
       'BusinessTravel_Non-Travel', 'BusinessTravel_Travel_Frequently',
       'BusinessTravel_Travel_Rarely', 'Department_Human Resources',
       'Department_Research & Development', 'Department_Sales',
       'EducationField_Human Resources', 'EducationField_Life Sciences',
       'EducationField_Marketing', 'EducationField_Medical',
       'EducationField_Other', 'EducationField_Technical Degree',
       'JobRole_Healthcare Representative', 'JobRole_Human Resources',
       'JobRole_Laboratory Technician', 'JobRole_Manager',
       'JobRole_Manufacturing Director', 'JobRole_Research Director',
       'JobRole_Research Scientist', 'JobRole_Sales Executive',
       'JobRole_Sales Representative', 'MaritalStatus_Divorced',
       'MaritalStatus_Married', 'MaritalStatus_Single']

# Create another list of predictor variables: Age, Country dummies (without the first country dummy) 
# [i for i in ad_data2.columns if i.startswith('Country')]: chooses all items in ad_data2.columns (var names)
# which start with 'Country'
# [i for i in ad_data2.columns if i.startswith('Country')][1:] -> add all countries but the first one (drop country at index 0)


###########predictors2 = ['Age']+[i for i in df1.columns if i.startswith('BusinessTravel')][1:]+[i for i in df1.columns if i.startswith('Department')][1:]+[i for i in df1.columns if i.startswith('EducationField')][1:]+[i for i in df1.columns if i.startswith('JobRole')][1:]+[i for i in df1.columns if i.startswith('MaritalStatus')][1:]


##-----BusinessTravel------###
##-----Department------###
##-----EducationField------###
##-----JobRole------###
##-----MaritalStatus------###




In [ ]:
# create dataframes for X (using Age only) and y variables 
X = df1[predictors1] # choose predictors1
y = df1['Attrition'] # choose target var

# see list of X variables 
# X.columns is the list of var names
# [i for i in X.columns]: choose all the items in X.columns (var names in X) in list
print('X variables:\n', [i for i in X.columns])


In [ ]:
X



In [ ]:
dfs = preprocessing.scale(X)
print(dfs)

In [ ]:
X
dfs





### Sklearn

In [ ]:
# split the dataset into training and test sets (e.g. Use 30% of the data as test data, 70% as training data)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=101)

# Define the type of model using function LogisticRegression()
model = LogisticRegression()

# Fit the model using training set
model.fit(df_train, y_train)


In [ ]:
# Predict the y values for the test set using function 'model.predict'
y_pred = model.predict(df_test)

# see predictions 
y_pred


In [ ]:
# calculate the confusion matrix for the test data using function 'confusion_matrix' in sklearn
# inputs: y_test, y_pred
# y_test: true y-values 
# y_pred: predicted y-values 
confusion_matrix_results = confusion_matrix(y_test, y_pred)

# print the counts of the confusion matrix 
print('confusion matrix: \n', confusion_matrix_results)

# print the metrics using function 'display_metrics' we wrote
display_metrics('Logistic Regression', y_pred, y_test)


### Statsmodels 


In [ ]:
# Estimate a logistic regression model on the data set with statsmodels 
# feed in all data (no spliting)
model = sm.Logit(y, X)
result = model.fit() 
result.summary2()


# Decision Tree

In [ ]:
# Create a list of predictor (x) variables: just Age
predictors1 = ['Age', 'DailyRate', 'DistanceFromHome', 'Education',
       'EmployeeCount', 'EmployeeNumber', 'EnvironmentSatisfaction', 'Gender',
       'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction',
       'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'OverTime',
       'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction',
               'StockOptionLevel', 'TotalWorkingYears',
       'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany',
       'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager',
       'BusinessTravel_Non-Travel', 'BusinessTravel_Travel_Frequently',
       'BusinessTravel_Travel_Rarely', 'Department_Human Resources',
       'Department_Research & Development', 'Department_Sales',
       'EducationField_Human Resources', 'EducationField_Life Sciences',
       'EducationField_Marketing', 'EducationField_Medical',
       'EducationField_Other', 'EducationField_Technical Degree',
       'JobRole_Healthcare Representative', 'JobRole_Human Resources',
       'JobRole_Laboratory Technician', 'JobRole_Manager',
       'JobRole_Manufacturing Director', 'JobRole_Research Director',
       'JobRole_Research Scientist', 'JobRole_Sales Executive',
       'JobRole_Sales Representative', 'MaritalStatus_Divorced',
       'MaritalStatus_Married', 'MaritalStatus_Single']

# Create another list of predictor variables: Age, Country dummies (without the first country dummy) 
# [i for i in ad_data2.columns if i.startswith('Country')]: chooses all items in ad_data2.columns (var names)
# which start with 'Country'
# [i for i in ad_data2.columns if i.startswith('Country')][1:] -> add all countries but the first one (drop country at index 0)


###########predictors2 = ['Age']+[i for i in df1.columns if i.startswith('BusinessTravel')][1:]+[i for i in df1.columns if i.startswith('Department')][1:]+[i for i in df1.columns if i.startswith('EducationField')][1:]+[i for i in df1.columns if i.startswith('JobRole')][1:]+[i for i in df1.columns if i.startswith('MaritalStatus')][1:]


##-----BusinessTravel------###
##-----Department------###
##-----EducationField------###
##-----JobRole------###
##-----MaritalStatus------###




In [ ]:
# Oversample and plot imbalanced dataset with ADASYN
from collections import Counter
from sklearn.datasets import make_classification
from imblearn.over_sampling import ADASYN
from matplotlib import pyplot
from numpy import where



# encoding
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder


In [ ]:
# Converting Dependent Variable DataType

##Boolean/Categorical Data(attrition, Gender, Overtime)

##---Attrition----##
# df['Attrition'] = df.Attrition.astype('bool') #convert var 'date'to a date data type in our dataframe 'df'

# df.loc[df["Attrition"] == "", "Attrition"] = 1
# df.loc[df["Attrition"] == "No", "Attrition"] = 0
#df1["Attrition"] = np.where(df["Attrition"] == "False", 0, 1)

##Nominal Data/Ordinal Data/ Interval Data/ Ratio Data (PerformanceRating, etc.))

# print(df.info())
# print(df)

isinstance(y, pd.DataFrame)
#[1470 rows x 33 columns]


In [ ]:
# create dataframes for X (using Age only) and y variables 
X = df1[predictors1] # choose predictors1
y = df1['Attrition'] # choose target var

# see list of X variables 
# X.columns is the list of var names
# [i for i in X.columns]: choose all the items in X.columns (var names in X) in list
print('X variables:\n', [i for i in X.columns])


In [ ]:

# test_size of 30%, random_state=101
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=101, stratify=y)


In [ ]:
# print(X_train)
y_train
y_train = pd.DataFrame(y_train)
print(y_train)
isinstance(y_train, pd.DataFrame)


In [ ]:
# le = LabelEncoder()

# le.fit(y_train)

# y_train = le.transform(y_train)
# y_test = le.transform(y_test)

# y_train
# y

In [ ]:
y_train['Attrition'].value_counts()

# y_train.value_counts



## Training a Decision Tree Model
- Use DecisionTreeClassifier in sklearn 
- Some paramters: 
    - criterion: The function to measure the quality of a split. Supported criteria are “gini” for the Gini impurity and “entropy” for the information gain.
    - max_depth: The maximum depth of the tree. If None, then nodes are expanded until all leaves are pure or until all leaves contain less than min_samples_split samples.
    - min_samples_leaf: The minimum number of samples required to be at a leaf node. 
    - max_features: The number of features to consider when looking for the best split:
    - see all parameters in the documentation: https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html

## Data Prep: Decision Tree

### 1. Unbalanced DV

In [ ]:
y_train



In [ ]:

print('Original dataset shape %s' % Counter(y))
ada = ADASYN(random_state=101)
X_res, y_res = ada.fit_resample(X, y)
# print('Resampled dataset shape %s' % Counter(y_res))




# # summarize class distribution
# counter = Counter(y)
# print(counter)
# # transform the dataset
# sm = ADASYN(random_state=101)
# X_res, y_res = sm.fit_resample(X_train, y_train)
# # summarize the new class distribution
# counter = Counter(y_res)
# print(counter)
# # scatter plot of examples by class label
# for label, _ in counter.items():
# 	row_ix = where(y_res == label)[0]
# 	pyplot.scatter(X_res[row_ix, 0], X_res[row_ix, 1], label=str(label))
# pyplot.legend()
# pyplot.show()




# def makeOverSamplesADASYN(X,y):
#  #input DataFrame
#  #X →Independent Variable in DataFrame\
#  #y →dependent Variable in Pandas DataFrame format
#  from imblearn.over_sampling import ADASYN 
#  sm = ADASYN()
#  X, y = sm.fit_sample(X, y)
#  return(X,y)



# #Apply Over Sampling
# print('Before Oversampling')
# print(sorted(Counter(y_train).items()))
# X_train, y_train = SMOTE().fit_resample(X_train, y_train)
# print('After Oversampling')
# print(sorted(Counter(y_train).items()))
# #Standard Scaler
# #scaler = StandardScaler()  
# #scaler.fit(X_train)  
# #X_train = scaler.transform(X_train)  
# #X_test = scaler.transform(X_test)  

In [ ]:
# train model, set criterion, max_depth=5 (no tree over 5 levels)
model_dt = DecisionTreeClassifier(criterion='entropy', random_state=101, max_depth=5)

# fit the model
model_dt.fit(X_train, y_train)


## Feature Importance


In [ ]:
# visualize x-vars by importance
importances = model_dt.feature_importances_  # extract importance metrics 
indices = np.argsort(importances)[::-1] # sorts the rows 

print("Feature ranking:")

feature_names = X_train.columns 

# create a dataframe using Pandas for the vars using pd.DataFrame() using feature names and importance metrics
fi = pd.DataFrame([feature_names[indices[0:10]], importances[indices][0:10]])
fi = fi.T
fi.columns = ['Feature', 'Total Reduction of Criterion']

print(fi)

# Plot the feature importances of the forest using seaborn
plt.figure()
plt.title("Feature importances")

sns.set_color_codes("pastel")
sns.barplot(x="Total Reduction of Criterion", y="Feature", data=fi, color="b")


## Predictions and Evaluation of Decision Tree

In [ ]:
# use predict function to make predictions 
y_pred = model_dt.predict(X_test)

In [ ]:
# calculate the confusion matrix for the test data 
confusion_matrix_results = confusion_matrix(y_test, y_pred)

# print the counts of the confusion matrix 
print('confusion matrix: \n', confusion_matrix_results)

# print the metrics 
display_metrics('Decision Tree', y_pred, y_test)


In [ ]:
# predict each instance's probability of survival for each individual using 'predict_proba' function
model_dt.predict_proba(X_train)


# Random Forest

## Training the Random Forest model
- Use RandomForestClassifier in sklearn 
- Some paramters: 
    - n_estimators: The number of trees in the forest.
    - max_depth: The maximum depth of the tree. If None, then nodes are expanded until all leaves are pure or until all leaves contain less than min_samples_split samples.
    - min_samples_leaf: The minimum number of samples required to be at a leaf node. 
    - max_features: The number of features to consider when looking for the best split:
    - see all parameters in the documentation: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html

In [ ]:
# train rf, set e.g. n_estimators = 500 (500 trees in the rf)
model_rf = RandomForestClassifier(n_estimators=500, random_state=101)

# fit model
model_rf.fit(X_train, y_train)


## Feature Importance

In [ ]:
# rank the features by importance 
importances = model_rf.feature_importances_
indices = np.argsort(importances)[::-1]

# Print the feature ranking
print("Feature ranking:")

feature_names = X_train.columns 

fi = pd.DataFrame([feature_names[indices[0:10]], importances[indices][0:10]])
fi = fi.T
fi.columns = ['Feature', 'Total Reduction of Criterion']

# Plot the feature importances 
plt.figure()
plt.title("Feature importances")

sns.set_color_codes("pastel")
sns.barplot(x="Total Reduction of Criterion", y="Feature", data=fi, color="b")


### Predictions and Evaluation

In [ ]:
y_pred = model_rf.predict(X_test)

In [ ]:
# calculate the confusion matrix for the test data 
confusion_matrix_results = confusion_matrix(y_test, y_pred)

# print the counts of the confusion matrix 
print('confusion matrix: \n', confusion_matrix_results)

# print the metrics 
display_metrics('Random Forest', y_pred, y_test)


In [ ]:
df.groupby('Attrition').sum().sort_values('Education') #1233


In [ ]:
df

Typically, a visual check is sufficient for determining normality. You can do this by making a histogram of your variable and looking for asymmetry (skewness) or outlying values. If you are comparing multiple groups for a numeric outcome variable (two-sample independent t-test or ANOVA), be sure to look at the distribution of the outcome variable for each group separately.



In [ ]:
# # import useful library
# import numpy as np
# from scipy.stats import shapiro
  

# # conduct the  Shapiro-Wilk Test
# shapiro(df)

In [ ]:
# ##QQ plots

# import math
# import numpy as np
# from scipy.stats import lognorm
# import statsmodels.api as sm
# import matplotlib.pyplot as plt

# #create Q-Q plot with 45-degree line added to plot
# fig = sm.qqplot(df, line='45')

# plt.show()

### Histogram

This is an important aspect that will be further discussed in this kernel and that is dealing with imbalanced dataset. 84% of employees did not quit the organization while 16% did leave the organization. Knowing that we are dealing with an imbalanced dataset will help us determine what will be the best approach to implement our predictive model.


# Structure of Data


# Label

In [ ]:
# Attrition - True/False 

In [ ]:
# Summary of Data

# Eplore and Describe Data

In [ ]:
# Create figure with 1 subplot (name the figure 'fig', and its axes 'ax')
fig, ax = plt.subplots(figsize=(30,12))

# Format the axes ('ax') of the plot 
#ax.plot(df.YearsAtCompany, df.DailyRate, marker='+', markersize=8) # set line marker
#ax.plot(df.YearsAtCompany, df.MonthlyIncome, marker='o', markersize=4) # set line marker
ax.plot(df.Age, df.DailyRate, marker='o', markersize=4) # set line marker

ax.set_xlabel('Age') # Notice the use of set_ to begin methods
ax.set_ylabel('Daily Income ($)')
ax.set_title('Monthly Income by Age')


In [ ]:
# box plot of different variables
plt.boxplot([df.DailyRate, df.MonthlyIncome])

# set x-axis labels (at indexes 1, 2)
plt.xticks(np.arange(50, 3), ('DailyRate', 'MonthlyIncome'))

# label for y-axis
plt.ylabel('Dollar ($)') 




# Standardization + PCA

In [ ]:
dfs.columns

In [ ]:
dfs


In [ ]:
dfs

In [ ]:
# create dummy variables for each country using function pd.get_dummies() for variable 'Country'
# add a prefix to names of dummies using 'prefix='Country'
# embed pd.get_dummies(ad_data2['Country'], prefix='Country') inside pd.conat()

# another way:
# countries = pd.get_dummies(ad_data2['Country'], prefix='Country')
# ad_data2 = pd.concat([ad_data2, countries],axis=1)

##-----BusinessTravel------###
dfs = pd.concat([dfs, pd.get_dummies(dfs['BusinessTravel'], prefix='BusinessTravel')],axis=1)
##-----Department------###
dfs = pd.concat([dfs, pd.get_dummies(dfs['Department'], prefix='Department')],axis=1)
##-----EducationField------###
dfs = pd.concat([dfs, pd.get_dummies(dfs['EducationField'], prefix='EducationField')],axis=1)
##-----JobRole------###
dfs = pd.concat([dfs, pd.get_dummies(dfs['JobRole'], prefix='JobRole')],axis=1)
##-----MaritalStatus------###
dfs = pd.concat([dfs, pd.get_dummies(dfs['MaritalStatus'], prefix='MaritalStatus')],axis=1)



In [ ]:
### now drop the original 'country' column (you don't need it anymore)


##-----BusinessTravel------###
dfs.drop(['BusinessTravel'],axis=1, inplace=True)
##-----Department------###
dfs.drop(['Department'],axis=1, inplace=True)

##-----EducationField------###
dfs.drop(['EducationField'],axis=1, inplace=True)

##-----JobRole------###
dfs.drop(['JobRole'],axis=1, inplace=True)

##-----MaritalStatus------###
dfs.drop(['MaritalStatus'],axis=1, inplace=True)


In [ ]:
dfs.columns()


In [ ]:
# # create the dataset with labels (y-variable, the written number in this case) and features (64 x-variables)
# # label: digit
# # 64 pixels -> 64 x-variables (each var is a pixel, and the value represents the darkness)
# features = pd.DataFrame(df1.data)
# labels = pd.DataFrame(df1.target, columns={'label'}) # label is the value of the target var
# data = pd.concat([labels, features], axis=1)
# data


In [ ]:
# Create a list of predictor (x) variables: just Age
predictors1 = ['Age', 'DailyRate', 'DistanceFromHome', 'Education',
       'EmployeeCount', 'EmployeeNumber', 'EnvironmentSatisfaction', 'Gender',
       'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction',
       'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'OverTime',
       'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction',
               'StockOptionLevel', 'TotalWorkingYears',
       'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany',
       'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager',
       'BusinessTravel_Non-Travel', 'BusinessTravel_Travel_Frequently',
       'BusinessTravel_Travel_Rarely', 'Department_Human Resources',
       'Department_Research & Development', 'Department_Sales',
       'EducationField_Human Resources', 'EducationField_Life Sciences',
       'EducationField_Marketing', 'EducationField_Medical',
       'EducationField_Other', 'EducationField_Technical Degree',
       'JobRole_Healthcare Representative', 'JobRole_Human Resources',
       'JobRole_Laboratory Technician', 'JobRole_Manager',
       'JobRole_Manufacturing Director', 'JobRole_Research Director',
       'JobRole_Research Scientist', 'JobRole_Sales Executive',
       'JobRole_Sales Representative', 'MaritalStatus_Divorced',
       'MaritalStatus_Married', 'MaritalStatus_Single']

# Create another list of predictor variables: Age, Country dummies (without the first country dummy) 
# [i for i in ad_data2.columns if i.startswith('Country')]: chooses all items in ad_data2.columns (var names)
# which start with 'Country'
# [i for i in ad_data2.columns if i.startswith('Country')][1:] -> add all countries but the first one (drop country at index 0)


###########predictors2 = ['Age']+[i for i in df1.columns if i.startswith('BusinessTravel')][1:]+[i for i in df1.columns if i.startswith('Department')][1:]+[i for i in df1.columns if i.startswith('EducationField')][1:]+[i for i in df1.columns if i.startswith('JobRole')][1:]+[i for i in df1.columns if i.startswith('MaritalStatus')][1:]


##-----BusinessTravel------###
##-----Department------###
##-----EducationField------###
##-----JobRole------###
##-----MaritalStatus------###




In [ ]:
# create dataframes for X (using Age only) and y variables 
features = dfs[predictors1] # choose predictors1
y = dfs['Attrition'] # choose target var

# see list of X variables 
# X.columns is the list of var names
# [i for i in X.columns]: choose all the items in X.columns (var names in X) in list
print('X variables:\n', [i for i in X.columns])


In [ ]:
features = preprocessing.scale(features)
features

In [ ]:
features = pd.DataFrame (features)


In [ ]:
features.info
# dfs
# 1470 rows × 52 columns



In [ ]:
features.columns =['Age', 'DailyRate', 'DistanceFromHome', 'Education',
       'EmployeeCount', 'EmployeeNumber', 'EnvironmentSatisfaction', 'Gender',
       'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction',
       'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'OverTime',
       'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction',
               'StockOptionLevel', 'TotalWorkingYears',
       'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany',
       'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager',
       'BusinessTravel_Non-Travel', 'BusinessTravel_Travel_Frequently',
       'BusinessTravel_Travel_Rarely', 'Department_Human Resources',
       'Department_Research & Development', 'Department_Sales',
       'EducationField_Human Resources', 'EducationField_Life Sciences',
       'EducationField_Marketing', 'EducationField_Medical',
       'EducationField_Other', 'EducationField_Technical Degree',
       'JobRole_Healthcare Representative', 'JobRole_Human Resources',
       'JobRole_Laboratory Technician', 'JobRole_Manager',
       'JobRole_Manufacturing Director', 'JobRole_Research Director',
       'JobRole_Research Scientist', 'JobRole_Sales Executive',
       'JobRole_Sales Representative', 'MaritalStatus_Divorced',
       'MaritalStatus_Married', 'MaritalStatus_Single']
# 1470 rows × 52 columns



In [ ]:
dfs

In [ ]:
features.info()

In [ ]:
labels = pd.DataFrame(y) # label is the value of the target var

In [ ]:
data = pd.concat([labels, features], axis=1)
data

# [1470 rows × 52 columns]

